In [2]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100
NCUBES = [3,4,5,6,7]

PCS = {
    'PC1':{'label':r'$\frac{\partial P}{\partial \widehat{\mathrm{RH}}} \geq 0$',
           'target_var':'rh','expected_sign':1,'other_vars':['thetae','thetaestar']},
    'PC2':{'label':r'$\frac{\partial P}{\partial \widehat{\theta_e}} \geq 0$',
           'target_var':'thetae','expected_sign':1,'other_vars':['rh','thetaestar']},
    'PC3':{'label':r'$\frac{\partial P}{\partial \widehat{\theta_e^*}} \leq 0$',
           'target_var':'thetaestar','expected_sign':-1,'other_vars':['rh','thetae']}}

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf = lfraw[valid]
obs = obsraw[valid]
landmask  = lf >= 0.5
oceanmask = lf < 0.5

FEATURES = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar}
print(f'Loaded {valid.sum()} valid samples ({landmask.sum()} land, {oceanmask.sum()} ocean)')

In [ ]:
def hypercube_test(targetvar,expectedsign,othervars,ncubes,mask=None):
    tvals = FEATURES[targetvar]
    ovals = [FEATURES[v] for v in othervars]
    if mask is None:
        mask = np.ones(len(tvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in ovals]
    bins = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(ovals,edges)]
    cubeidx = bins[0]*ncubes + bins[1]
    nsatisfied,ntested = 0,0
    for cidx in range(ncubes**2):
        sel = mask & (cubeidx==cidx)
        if sel.sum() < MINCUBESAMPLES:continue
        x,y = tvals[sel],obs[sel]
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expectedsign>=0 and slope>=0) or (expectedsign<0 and slope<=0):
            nsatisfied += 1
    return nsatisfied,ntested

rows = []
for pcname,pc in PCS.items():
    for region,rmask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
        pcts = []
        for n in NCUBES:
            sat,tot = hypercube_test(pc['target_var'],pc['expected_sign'],pc['other_vars'],n,rmask)
            pcts.append(sat/max(tot,1)*100)
        rows.append({'PC':pcname,'Region':region,'Average':np.mean(pcts)})

era5df = pd.DataFrame(rows).pivot(index='Region',columns='PC',values='Average')
era5df = era5df.reindex(index=['Land','Ocean','All'],columns=['PC1','PC2','PC3'])
era5df.columns = [PCS[c]['label'] for c in era5df.columns]
display(era5df.style.format('{:.1f}%').set_caption(
    f'Table S2: ERA5 constraint satisfaction (%, {SPLIT} set, hypercube test, min {MINCUBESAMPLES} samples/cube)'))

In [ ]:
def hypercube_details(targetvar,expectedsign,othervars,ncubes,mask=None):
    tvals = FEATURES[targetvar]
    ovals = [FEATURES[v] for v in othervars]
    if mask is None:
        mask = np.ones(len(tvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in ovals]
    bins = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(ovals,edges)]
    cubeidx = bins[0]*ncubes + bins[1]
    results = []
    for cidx in range(ncubes**2):
        sel = mask & (cubeidx==cidx)
        if sel.sum() < MINCUBESAMPLES:continue
        x,y = tvals[sel],obs[sel]
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        satisfied = (expectedsign>=0 and slope>=0) or (expectedsign<0 and slope<=0)
        results.append({'slope':slope,'satisfied':satisfied,'mean_precip':y.mean(),
                        othervars[0]:ovals[0][sel].mean(),othervars[1]:ovals[1][sel].mean()})
    return results

cubedata = []
for n in NCUBES:
    cubedata.extend(hypercube_details('rh',1,['thetae','thetaestar'],n,oceanmask))
slopes = np.array([c['slope'] for c in cubedata])
satisfied = np.array([c['satisfied'] for c in cubedata])
meanprecip = np.array([c['mean_precip'] for c in cubedata])
meante = np.array([c['thetae'] for c in cubedata])
meantes = np.array([c['thetaestar'] for c in cubedata])

nsat = satisfied.sum()
nviol = (~satisfied).sum()

fig,axs = pplt.subplots(ncols=3,figwidth=7,refheight=2.2,sharey=False)

slopebins = np.linspace(min(slopes.min(),-0.5),max(slopes.max(),2),35)
axs[0].hist(slopes[satisfied],bins=slopebins,color='#2355a1',alpha=0.7,label=f'Satisfies ({nsat})')
axs[0].hist(slopes[~satisfied],bins=slopebins,color='#D42028',alpha=0.7,label=f'Violates ({nviol})')
axs[0].axvline(0,color='k',lw=0.8,ls='--')
axs[0].legend(loc='ur',ncols=1,fontsize=7)
axs[0].format(xlabel=r'Slope $\Delta P / \Delta \widehat{\mathrm{RH}}$',ylabel='Hypercube count',
              title='Slope magnitude')

axs[1].scatter(meante[satisfied],meantes[satisfied],color='#2355a1',marker='o',s=12,alpha=0.5,label='Satisfies')
axs[1].scatter(meante[~satisfied],meantes[~satisfied],color='#D42028',marker='x',s=18,alpha=0.8,label='Violates',zorder=5)
axs[1].legend(loc='ur',ncols=1,fontsize=7)
axs[1].format(xlabel=r'Mean $\widehat{\theta}_e$',ylabel=r'Mean $\widehat{\theta}_e^*$',
              title='Thermodynamic regime')

precipbins = np.linspace(0,max(meanprecip.max(),1),30)
axs[2].hist(meanprecip[satisfied],bins=precipbins,color='#2355a1',alpha=0.7,label='Satisfies')
axs[2].hist(meanprecip[~satisfied],bins=precipbins,color='#D42028',alpha=0.7,label='Violates')
axs[2].legend(loc='ur',ncols=1,fontsize=7)
axs[2].format(xlabel='Mean precipitation (mm)',ylabel='Hypercube count',
              title='Precipitation regime')

axs.format(abc=True,titleloc='l',grid=False)
pplt.show()
fig.save('../figs/fig_pc1_ocean_diagnostic.jpg')

In [ ]:
regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']))
             for _,row in regdf.iterrows()}

atm = REGISTRY['sr_atm_eq']['constants']
a_atm,b_atm,c_atm = atm['a'],atm['b'],atm['c']
arg = thetae - b_atm*thetaestar - c_atm
rhbranch = rh >= arg
dz_dthetae_atm = np.where(~rhbranch,3*a_atm*arg**2,0.0)

for eqname in ['sr_all_eq','sr_all_pc_eq']:
    c = REGISTRY[eqname]['constants']
    if eqname=='sr_all_eq':
        dz = dz_dthetae_atm + (c['b']-lf)**3
    else:
        dz = dz_dthetae_atm + (c['b']-lf)**3 - (c['b']-1)**3
    sat = dz >= 0
    print(f"{eqname} PC2 satisfaction:")
    print(f"  Land (LF>=0.5): {np.mean(sat[landmask])*100:.1f}%")
    print(f"  Ocean (LF<0.5): {np.mean(sat[oceanmask])*100:.1f}%")
    print(f"  All:            {np.mean(sat)*100:.1f}%\n")